# Exercise 09 — Classifying images with a convolutional network

## What you are about to see

[Exercise 05](q05_binary_classification.ipynb) trained a network on a table: 108 numbers
per person, and the *order* of those numbers meant nothing. **An image is different.** Its
784 pixels sit on a grid, and what makes a `7` a `7` is a *local pattern* — a stroke, a
corner — that can appear anywhere on that grid.

This notebook trains a small **convolutional network** on handwritten digits (MNIST): a
network that slides the same few filters over every position of the image. The training
loop is the same four lines as always (`zero_grad` → loss → `backward` → `step`); what
changes is the layer. Then we look inside it — what its filters respond to, where it
fails — and finally we fool it.

Everything follows *Lecture V — Redes Convolucionais*; each section names the part it
puts to work. The whole notebook takes about **two minutes**, most of it in the CNN's training
cell and the attack at the end.

In [ ]:
# Run me first: make ``dlcb`` and ``datasets`` importable whether Jupyter was
# started in the project root or inside ``exercises/``.
import pathlib
import sys
import time

ROOT = pathlib.Path.cwd()
if not (ROOT / "dlcb").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np

import datasets
from dlcb import engine as cpu
from dlcb import nn, optim
from dlcb.conv import Conv2d, Flatten, MaxPool2d, im2col
from dlcb.loss import cross_entropy

print("ready — engine imported from", ROOT)

## 1. An image is a matrix

MNIST is 70 000 handwritten digits, $28 \times 28$ pixels in grey levels, each labelled
0–9: 60 000 to train on, and 10 000 written by *other people* to test on (LeCun et al.,
1998). `datasets.load_mnist` hands each image over as a matrix of intensities in
$[0, 1]$ — 0 is background, 1 is ink — with one channel (Lecture V, *Uma imagem em níveis
de cinza, pixel a pixel*). The layout is **channels-last**, `(batch, H, W, C)`: the
lecture's $H \times W \times C_{\text{in}}$ with a batch axis in front.

A pure-NumPy engine handles a few hundred images per second, so we train on the first
`N_TRAIN` digits of the training split, keep the next `N_VAL` aside to watch training,
and measure the final result on the full test split.

In [ ]:
N_TRAIN = 10_000        # digits to train on (of 60 000)
N_VAL = 2_000           # the next ones, to monitor training
BATCH = 64              # digits per Adam step
EPOCHS = 4              # passes over the training digits
LR = 3e-3               # Adam learning rate

In [ ]:
cpu.set_seed(0)

pool = datasets.load_mnist("train", limit=N_TRAIN + N_VAL)
x_train, y_train = pool.images[:N_TRAIN], pool.labels[:N_TRAIN]
x_val, y_val = pool.images[N_TRAIN:], pool.labels[N_TRAIN:]
test = datasets.load_mnist("test")

print(f"train {x_train.shape}   val {x_val.shape}   test {test.images.shape}")
print("digits per class in train:", np.bincount(y_train).tolist())

Every picture in this notebook is drawn by one helper. It shows 2-D maps side by side:
intensities in grey (darker = more ink), or, with `signed=True`, values that can be
negative — filter weights, perturbations — in blue (negative) and red (positive), with
white at zero. `matplotlib` only draws; nothing is computed with it.

In [ ]:
def plot_maps(maps: List[np.ndarray], titles: List[str], signed: bool = False,
              cols: int = 8) -> None:
    """Draw 2-D maps in a grid, each under its title."""
    rows = -(-len(maps) // cols)
    cols = min(cols, len(maps))
    fig, axes = plt.subplots(rows, cols, figsize=(1.7 * cols, 1.9 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, m, title in zip(axes.flat, maps, titles):
        if signed:
            v = np.abs(m).max() or 1.0
            ax.imshow(m, cmap="RdBu_r", vmin=-v, vmax=v)
        else:
            ax.imshow(m, cmap="gray_r", vmin=0.0, vmax=max(1.0, m.max()))
        ax.set_title(title, fontsize=9)
    plt.tight_layout()
    plt.show()


plot_maps([x_train[i, :, :, 0] for i in range(8)], [f"label {y_train[i]}" for i in range(8)])

## 2. A convolution is one matrix product

A convolutional layer keeps the grid. It slides a small filter — here $5 \times 5$ — over
every position, and the filter's weights are **the same at every position**
(Lecture V, *Localidade e compartilhamento*). The lecture's central section
(*Implementação por im2col e uma multiplicação matricial*) shows that the whole layer is

$$
\widetilde{\mathbf{X}} = \begin{bmatrix}\mathbf{1}^{\top}\\ \mathbf{X}_{\mathrm{col}}\end{bmatrix}
\in \mathbb{R}^{(K+1)\times N},
\qquad
\boxed{\;\mathbf{Z} = \mathbf{W}^{\top}\widetilde{\mathbf{X}}\;},
\qquad
\mathbf{W} \in \mathbb{R}^{(K+1)\times F},
$$

where column $p$ of $\mathbf{X}_{\mathrm{col}}$ is the window around output position $p$,
flattened ($K = C_{\text{in}} k_h k_w$ numbers), and column $f$ of $\mathbf{W}$ is filter
$f$ with its bias on top. That is **the dense layer's equation** from Lecture I, with one
change of meaning: a column of the input is no longer a sample but the *neighbourhood of
one position*. So in `dlcb`, `Conv2d` is literally `im2col` followed by an `nn.Linear` —
and needs no backward rule of its own: the engine's `@` gives
$\partial\mathcal{L}/\partial\mathbf{W} = \widetilde{\mathbf{X}}\mathbf{D}^{\top}$, and the
gather of `im2col` sums the gradients of every copy of a pixel (the lecture's `col2im`).

Look at the shapes on one digit:

In [ ]:
digit = cpu.Tensor(x_train[:1])                         # (1, 28, 28, 1)
x_col, (h_out, w_out) = im2col(digit, kernel=5)
conv = Conv2d(in_channels=1, out_channels=8, kernel_size=5)

print(f"image      {digit.shape}")
print(f"X_col      {x_col.shape}     K = 1·5·5 = 25 values per window, N = {h_out}·{w_out} windows")
print(f"W          {conv.weight.shape}       (K+1) × F: bias row + one column per filter")
print(f"output     {conv(digit).shape}  (batch, H_out, W_out, F)")
print(f"\nX_col holds {x_col.data.size:,} numbers for an image of {digit.data.size}: "
      f"im2col trades memory for one big matrix product.")

### A filter is a feature detector

Before learning anything, set two filters by hand. A filter answers one question at every
position (Lecture V, *Um filtro, uma característica*): here, "is there a **vertical** edge
here?" and "is there a **horizontal** edge here?". Each is the $3 \times 3$ Sobel pattern;
the ReLU keeps only the positive answers.

In [ ]:
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)   # vertical edges
edges = Conv2d(1, 2, kernel_size=3, padding=1)          # padding 1: keep 28 × 28
edges.weight.data[0] = 0.0                              # no bias
edges.weight.data[1:, 0] = sobel_x.ravel()              # filter 0
edges.weight.data[1:, 1] = sobel_x.T.ravel()            # filter 1: the transpose

maps = edges(cpu.Tensor(x_train[:1])).relu().data[0]    # (28, 28, 2)
plot_maps([sobel_x, sobel_x.T], ["filter 0", "filter 1"], signed=True)
plot_maps([x_train[0, :, :, 0], maps[:, :, 0], maps[:, :, 1]],
          ["the digit", "vertical edges", "horizontal edges"])

Training does the same thing, except that nobody writes the filters: they start random
and gradient descent chooses them. At the end we will look at what it chose.

## 3. The network: convolution, pooling, then a dense head

A classic small design, in the spirit of LeCun et al. (1998): two rounds of
*convolution → ReLU → max pooling*, then one dense layer on top (Lecture V, *Uma CNN
completa, de ponta a ponta*). **Max pooling** keeps the largest value of each $2 \times 2$
block: it halves the resolution and makes the answer "is the feature here?" tolerant to a
shift of a pixel (*Pooling e os limites da analogia matricial*). It has no parameters.

Each output size comes from the lecture's formula (*A fórmula das dimensões*),
$H_{\text{out}} = \lfloor (H + 2P - k)/S \rfloor + 1$:

| layer | output | parameters |
|---|---|---|
| input | $28 \times 28 \times 1$ | — |
| `Conv2d(1, 8, 5)` + ReLU | $24 \times 24 \times 8$ | $(25 + 1) \cdot 8 = 208$ |
| `MaxPool2d(2)` | $12 \times 12 \times 8$ | 0 |
| `Conv2d(8, 16, 5)` + ReLU | $8 \times 8 \times 16$ | $(200 + 1) \cdot 16 = 3\,216$ |
| `MaxPool2d(2)` | $4 \times 4 \times 16$ | 0 |
| `Flatten` + `Linear(256, 10)` | 10 logits | $257 \cdot 10 = 2\,570$ |

The second convolution's filters are $5 \times 5 \times 8$: each covers **all eight**
feature maps of the first layer (Lecture V, *Convolução 2D, 2D multicanal e 3D*), so it
detects combinations of strokes — corners, loops, crossings.

In [ ]:
class DigitCNN(nn.Module):
    """Two (Conv -> ReLU -> MaxPool) stages and a dense head: about 6 000 parameters."""

    def __init__(self) -> None:
        self.conv1 = Conv2d(1, 8, kernel_size=5)        # 28×28×1  -> 24×24×8
        self.conv2 = Conv2d(8, 16, kernel_size=5)       # 12×12×8  -> 8×8×16
        self.pool = MaxPool2d(2)                        # halves H and W
        self.flatten = Flatten()                        # 4×4×16   -> (256, batch)
        self.head = nn.Linear(4 * 4 * 16, 10)

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        h = self.pool(self.conv1(x).relu())             # (batch, 12, 12, 8)
        h = self.pool(self.conv2(h).relu())             # (batch, 4, 4, 16)
        return self.head(self.flatten(h)).T             # (batch, 10) logits


def count_parameters(model: nn.Module) -> int:
    return sum(p.data.size for p in model.parameters())


cnn = DigitCNN()
print(f"CNN: {count_parameters(cnn):,} parameters")

## 4. Training

Ten classes, so the output is ten logits and the loss is the softmax cross-entropy of
Exercise 05 — nothing about the loss is specific to images. We step with Adam on
mini-batches of `BATCH` digits, reshuffled every epoch, and report three things per epoch:
the mean training loss, the accuracy on the `N_VAL` held-out digits, and the cumulative
training cost in FLOPs, as counted by the engine.

Measuring accuracy needs only a forward pass, so `predict` runs it inside
`cpu.no_grad()`: the ops then record no graph. That is not just a saving. Every recorded
node is kept alive by a reference cycle until `backward()` breaks it, and a forward pass
that is never followed by a backward — evaluating 10 000 test digits, say — would leave
gigabytes of feature maps waiting for Python's garbage collector.

In [ ]:
def predict(model: nn.Module, images: np.ndarray, chunk: int = 500) -> np.ndarray:
    """Predicted class of every image, in chunks, without building a graph."""
    with cpu.no_grad():
        return np.concatenate([
            model(cpu.Tensor(images[i:i + chunk])).data.argmax(axis=1)
            for i in range(0, len(images), chunk)
        ])


def train(model: nn.Module, lr: float) -> int:
    """Mini-batch Adam on softmax cross-entropy; returns the training FLOPs."""
    opt = optim.Adam(model.parameters(), lr=lr)
    cpu.reset_flops()
    started = time.time()
    for epoch in range(1, EPOCHS + 1):
        order = np.random.permutation(N_TRAIN)
        losses = []
        for start in range(0, N_TRAIN, BATCH):
            rows = order[start:start + BATCH]
            opt.zero_grad()
            loss = cross_entropy(model(cpu.Tensor(x_train[rows])), y_train[rows])
            loss.backward()
            opt.step()
            losses.append(float(loss.data))
        flops = cpu.flop_count()                        # before the validation pass
        val_acc = (predict(model, x_val) == y_val).mean()
        print(f"   epoch {epoch}/{EPOCHS}   loss {np.mean(losses):.3f}   "
              f"val accuracy {val_acc:.4f}   [{time.time() - started:4.0f} s, "
              f"{flops / 1e9:5.1f} GFLOP]")
    return flops

Training takes about **a minute**. The network is small, but it does a lot of
arithmetic: every filter is applied at every position.

In [ ]:
print("Training the CNN (mini-batch Adam):")
cnn_flops = train(cnn, LR)

## 5. Read the result

In [ ]:
cnn_pred = predict(cnn, test.images)
cnn_acc = (cnn_pred == test.labels).mean()

print(f"test accuracy:  {cnn_acc:.4f}   ({(cnn_pred != test.labels).sum()} of {len(test)} digits wrong)")
print(f"parameters:     {count_parameters(cnn):,}")
print(f"training cost:  {cnn_flops / 1e9:.1f} GFLOP")
print(f"\n(always answering the commonest digit would score {np.bincount(test.labels).max() / len(test):.4f})")

Three things to read in those numbers:

1. **It learned.** About 97 % of digits written by people it never saw are read
   correctly, against 11 % for always answering the commonest digit.
2. **With very few parameters.** About 6 000 numbers in all. That is weight sharing: a
   $5 \times 5$ filter is 26 numbers wherever on the image it is applied, so the count
   does not depend on the size of the image.
3. **But not with little arithmetic.** Each of those few weights is used at every
   position — all $24 \times 24 = 576$ of them in the first layer — so training cost tens
   of GFLOP. Parameters and arithmetic are different budgets (Lecture V, *Memória de
   parâmetros $\neq$ memória de ativações*).

### Where the CNN is wrong

A **confusion matrix** counts, for every true digit (row), how often each digit was
predicted (column); the diagonal holds the correct answers. Below it, a few of the test
digits the CNN got wrong — many are ambiguous even to a human.

In [ ]:
def confusion_matrix(truth: np.ndarray, pred: np.ndarray) -> np.ndarray:
    m = np.zeros((10, 10), dtype=int)
    np.add.at(m, (truth, pred), 1)
    return m


m = confusion_matrix(test.labels, cnn_pred)
print("true \\ predicted " + "".join(f"{d:>6}" for d in range(10)))
for d in range(10):
    print(f"{d:>16} " + "".join(f"{v:>6}" for v in m[d]))

wrong = np.flatnonzero(cnn_pred != test.labels)[:8]
plot_maps([test.images[i, :, :, 0] for i in wrong],
          [f"true {test.labels[i]}, said {cnn_pred[i]}" for i in wrong])

### What the first layer learned

The eight learned filters of `conv1` are column $f$ of $\mathbf{W}$, without its bias
row, reshaped to $5 \times 5$: `cnn.conv1.weight.data[1:, f].reshape(5, 5)` — the first
row of pictures below, blue negative and red positive. They are not as tidy as the Sobel
pair: gradient descent had no reason to make them tidy, only useful. What they **do** is
easier to read: the second row is their eight feature maps on one test digit, after the
ReLU — the same kind of picture as the hand-made edge maps of Section 2. Different filters
light up on different parts of the strokes, like oriented edge detectors found by
gradient descent instead of written by hand.

In [ ]:
digit = test.images[:1]
with cpu.no_grad():
    maps = cnn.conv1(cpu.Tensor(digit)).relu().data[0]            # (24, 24, 8)
filters = [cnn.conv1.weight.data[1:, f].reshape(5, 5) for f in range(8)]
plot_maps(filters, [f"filter {f}" for f in range(8)], signed=True)
plot_maps([maps[:, :, f] for f in range(8)], [f"map {f} (digit {test.labels[0]})" for f in range(8)])

## 6. Fooling the trained network

The CNN reads 97 % of the test digits correctly. Now the problem: take a digit it reads
correctly, **add a faint pattern to it** — so faint that a person still sees the same
digit — and the network changes its answer.

The pattern is not random. It is computed from the network itself, with the tool used to
train it: the gradient. Training asks *how should the weights move to lower the loss?*;
here we ask *how should the **pixels** move to raise it?* It is the same `backward()` —
the engine computes $\partial\mathcal{L}/\partial\mathbf{x}$ for the input on the way to the
weights anyway, because the input is a `Tensor` like any other. We just read it at a
different leaf.

**The fast gradient sign method** (Goodfellow, Shlens and Szegedy, 2015) moves every pixel
by the same small amount $\varepsilon$ in the direction that increases the loss:

$$
\mathbf{x}_{\mathrm{adv}} = \operatorname{clip}_{[0,1]}\!\Bigl(\mathbf{x} + \varepsilon\,
\operatorname{sign}\bigl(\nabla_{\mathbf{x}}\,\mathcal{L}(\mathbf{x}, y)\bigr)\Bigr).
$$

No pixel changes by more than $\varepsilon$. The **iterative** version (Kurakin, Goodfellow
and Bengio, 2017) takes several smaller steps of the same kind, recomputing the gradient
each time and never letting a pixel drift more than $\varepsilon$ from the original.

Why can so small a change matter? The network is piecewise linear (ReLU, max pooling,
linear layers), and a linear score $\mathbf{w}^{\top}\mathbf{x}$ moved by
$\varepsilon\,\operatorname{sign}(\mathbf{w})$ changes by $\varepsilon\lVert\mathbf{w}\rVert_1$:
a sum over all 784 pixels. Each pixel's change is invisible; together they add up. These
*adversarial examples* were first reported by Szegedy et al. (2014).

> **A note on names.** This attacks the network *after* training, at the moment it is
> used. Its sibling, **data poisoning**, tampers with the *training* images instead, so that
> the network learns the wrong thing (Gu, Dolan-Gavitt and Garg, 2017). Here the network was
> trained on clean data and never saw a modified image.

In [ ]:
def input_gradient(model: nn.Module, images: np.ndarray, labels: np.ndarray,
                   chunk: int = 250) -> np.ndarray:
    """dL/dx for every image: the same backward() as training, read at the input."""
    grads = []
    for i in range(0, len(images), chunk):
        x = cpu.Tensor(images[i:i + chunk])               # a leaf, like the weights
        cross_entropy(model(x), labels[i:i + chunk]).backward()
        grads.append(x.grad)
    model.zero_grad()                                     # the weights' gradients are not used
    return np.concatenate(grads)


def fgsm(model: nn.Module, images: np.ndarray, labels: np.ndarray, eps: float) -> np.ndarray:
    """One step of size eps along the sign of the input gradient."""
    return np.clip(images + eps * np.sign(input_gradient(model, images, labels)), 0.0, 1.0)


def iterative_attack(model: nn.Module, images: np.ndarray, labels: np.ndarray, eps: float,
                     steps: int = 10) -> np.ndarray:
    """Several steps of size eps/4, each kept within eps of the original pixels."""
    x = images.copy()
    for _ in range(steps):
        x = x + (eps / 4) * np.sign(input_gradient(model, x, labels))
        x = np.clip(np.clip(x, images - eps, images + eps), 0.0, 1.0)
    return x

In [ ]:
N_ATTACK = 500                                          # test digits to attack
x_clean, y_clean = test.images[:N_ATTACK], test.labels[:N_ATTACK]

print(f"{'eps':>6}{'CNN, one step':>16}{'CNN, iterative':>17}")
for eps in (0.0, 0.05, 0.1, 0.15, 0.2):
    one = (predict(cnn, fgsm(cnn, x_clean, y_clean, eps)) == y_clean).mean()
    many = (predict(cnn, iterative_attack(cnn, x_clean, y_clean, eps)) == y_clean).mean()
    print(f"{eps:>6.2f}{one:>16.4f}{many:>17.4f}")

Look at what the attack did to a few digits the CNN had read correctly. Top row: the
original, with the network's answer and its confidence. Middle row: the perturbation —
red adds ink, blue removes it, never more than $\varepsilon$ per pixel (the background can
only gain ink, since it is already 0). Bottom row: original plus perturbation, and what the
network says now. The attack takes about 40 seconds: ten gradients per digit, for each
$\varepsilon$.

In [ ]:
EPS = 0.15
correct = np.flatnonzero(predict(cnn, x_clean) == y_clean)[:8]
x_adv = iterative_attack(cnn, x_clean[correct], y_clean[correct], EPS)
with cpu.no_grad():
    p_clean = cnn(cpu.Tensor(x_clean[correct])).softmax(axis=-1).data
    p_adv = cnn(cpu.Tensor(x_adv)).softmax(axis=-1).data

plot_maps([x_clean[i, :, :, 0] for i in correct],
          [f"{p.argmax()} ({p.max():.0%})" for p in p_clean])
plot_maps([x_adv[k, :, :, 0] - x_clean[i, :, :, 0] for k, i in enumerate(correct)],
          [f"perturbation" for _ in correct], signed=True)
plot_maps([x_adv[k, :, :, 0] for k in range(len(correct))],
          [f"{p.argmax()} ({p.max():.0%})" for p in p_adv])

Read the table and the pictures together:

- **The accuracy collapses.** At $\varepsilon = 0.15$ one step takes the CNN from about
  0.98 to about 0.26 on these digits, and ten small steps to about 0.04 — on digits that
  any reader of the bottom row still recognises at a glance.
- **The network is wrong with confidence.** It does not hesitate between two answers; it
  gives the wrong digit with 70–100 % probability. Its confidence says nothing about
  whether the input was tampered with.
- **Nothing in the network changed.** The weights are the ones that scored 97 % in
  Section 5. The attack only read the gradient that training computes anyway — the same
  `backward()`, followed to a different leaf.

---

**What to carry forward.**

- A convolutional layer is the dense layer's $\mathbf{Z} = \mathbf{W}^{\top}\widetilde{\mathbf{X}}$
  with the columns reinterpreted as windows. Everything else — the loss, `backward()`,
  Adam — is unchanged, which is why `dlcb.conv` added no rule to the engine.
- **High accuracy is not robustness.** The same gradient that trains the network tells an
  attacker, pixel by pixel, how to change its answer without changing what a person sees.
- **Parameters and arithmetic are different budgets** (Lecture V, *Memória de parâmetros
  $\neq$ memória de ativações*): the CNN stores few numbers and spends many FLOPs. Weight
  sharing buys the first; the second is the price of applying every filter everywhere.

**Try it.**

1. Remove the second convolution (`conv1 → pool → Flatten → Linear(12·12·8, 10)`). How
   many parameters does the head need now, and what happens to accuracy?
2. Replace `MaxPool2d` by `AvgPool2d` (`from dlcb.conv import AvgPool2d`). Lecture V says
   average pooling is linear and max pooling is not; does it matter here?